# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shreshth114/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

### Baseline rule

I will prioritize pages that have meaningful search visibility but appear to be weaker candidates for continued performance based on staleness and search performance.

The score combines:
- content staleness
- impressions/traffic opportunity
- average search position

Higher scores receive higher priority.

Reason codes:
- STALE_HIGH_VOLUME: old content with meaningful search visibility
- STALE_POOR_POSITION: old content with a weaker average position
- HIGH_VOLUME_OPPORTUNITY: meaningful search visibility with room for improvement
- GENERAL_REFRESH: lower-confidence refresh candidate

The action label is `refresh_review`.

This is a directional decision-support rule, not a causal claim.

In [12]:
import pandas as pd
import numpy as np
import os

data_path = "/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(data_path)

print("Dataset loaded:", df.shape)
print("Columns:", df.columns.tolist())

df["content_age_days"] = pd.to_numeric(
    df["content_age_days"], errors="coerce"
)

df["days_since_last_update"] = pd.to_numeric(
    df["days_since_last_update"], errors="coerce"
)

df["impressions_90d"] = pd.to_numeric(
    df["impressions_90d"], errors="coerce"
)

df["avg_position"] = pd.to_numeric(
    df["avg_position"], errors="coerce"
)

df = df[
    (df["impressions_90d"] > 0) &
    (df["content_age_days"] >= 90)
].copy()

print("Rows available for baseline:", len(df))

Dataset loaded: (30000, 44)
Columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']
Rows available for baseline: 30000


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [13]:
df["score"] = (
    (df["content_age_days"] >= 365).astype(int) * 3
    + (df["content_age_days"] >= 180).astype(int) * 2
    + (df["impressions_90d"] >= 500).astype(int) * 2
    + (df["impressions_90d"] >= 1000).astype(int) * 1
    + (df["avg_position"] >= 10).astype(int) * 2
)

def get_reason(row):
    if row["content_age_days"] >= 365 and row["impressions_90d"] >= 500:
        return "STALE_HIGH_VOLUME"
    elif row["content_age_days"] >= 365 and row["avg_position"] >= 10:
        return "STALE_POOR_POSITION"
    elif row["impressions_90d"] >= 1000 and row["avg_position"] >= 10:
        return "HIGH_VOLUME_OPPORTUNITY"
    else:
        return "GENERAL_REFRESH"

df["reason_code"] = df.apply(get_reason, axis=1)
df["action"] = "refresh_review"

queue = df.sort_values(
    ["score", "impressions_90d", "content_age_days"],
    ascending=[False, False, False]
).copy()

output_cols = [
    "score",
    "reason_code",
    "action",
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position"
]

queue = queue[output_cols].reset_index(drop=True)

os.makedirs(
    "/content/flyrank-ml-internship/work/outputs",
    exist_ok=True
)

output_path = "/content/flyrank-ml-internship/work/outputs/baseline_action_score.csv"

queue.to_csv(output_path, index=False)

print("Queue written to:", output_path)
print("Queue size:", len(queue))
print(queue.head(10).to_string(index=False))

Queue written to: /content/flyrank-ml-internship/work/outputs/baseline_action_score.csv
Queue size: 30000
 score       reason_code         action  content_age_days  days_since_last_update  impressions_90d  avg_position
    10 STALE_HIGH_VOLUME refresh_review               445                      25           173450          22.6
    10 STALE_HIGH_VOLUME refresh_review               445                      25            60021          22.6
    10 STALE_HIGH_VOLUME refresh_review               445                     104            59474          35.2
    10 STALE_HIGH_VOLUME refresh_review               445                      25            52313          10.0
    10 STALE_HIGH_VOLUME refresh_review               445                      25            51945          24.1
    10 STALE_HIGH_VOLUME refresh_review               445                     104            50865          28.4
    10 STALE_HIGH_VOLUME refresh_review               445                      20            48819     


### Top-20 review

The following review treats the baseline as decision-support. A high score does not prove that refreshing a page will improve performance. Each pick should be checked against the actual page and business context before action.

In [14]:
top20 = queue.head(20).copy()

def confidence_note(score):
    if score >= 7:
        return "Higher confidence because multiple signals agree."
    elif score >= 4:
        return "Moderate confidence because fewer signals agree."
    else:
        return "Lower confidence; manual review is important."

def wrong_if(row):
    if row["reason_code"] == "STALE_HIGH_VOLUME":
        return "Wrong if the page is intentionally evergreen or already scheduled for refresh."
    elif row["reason_code"] == "STALE_POOR_POSITION":
        return "Wrong if the position reflects a query mix that is not relevant to the page."
    elif row["reason_code"] == "HIGH_VOLUME_OPPORTUNITY":
        return "Wrong if the page has high visibility but no realistic content opportunity."
    else:
        return "Wrong if the page has a valid reason not to be refreshed."

top20["confidence_note"] = top20["score"].apply(confidence_note)
top20["what_would_make_it_wrong"] = top20.apply(wrong_if, axis=1)

print(
    top20[
        [
            "action",
            "reason_code",
            "confidence_note",
            "what_would_make_it_wrong"
        ]
    ].to_string(index=False)
)

        action       reason_code                                   confidence_note                                                       what_would_make_it_wrong
refresh_review STALE_HIGH_VOLUME Higher confidence because multiple signals agree. Wrong if the page is intentionally evergreen or already scheduled for refresh.
refresh_review STALE_HIGH_VOLUME Higher confidence because multiple signals agree. Wrong if the page is intentionally evergreen or already scheduled for refresh.
refresh_review STALE_HIGH_VOLUME Higher confidence because multiple signals agree. Wrong if the page is intentionally evergreen or already scheduled for refresh.
refresh_review STALE_HIGH_VOLUME Higher confidence because multiple signals agree. Wrong if the page is intentionally evergreen or already scheduled for refresh.
refresh_review STALE_HIGH_VOLUME Higher confidence because multiple signals agree. Wrong if the page is intentionally evergreen or already scheduled for refresh.
refresh_review STALE_HIGH_VO

### Weak picks

The weakest picks are rows with relatively low baseline scores. They may still be useful, but the evidence for prioritizing them is weaker, so they should receive manual review before action.

### Leakage check

The baseline uses only fields available in the starter dataset before any future outcome is considered. It does not use a future-month label, product flags, or a label-derived column. The baseline is therefore intended as a pre-outcome directional ranking.

In [15]:
weak_picks = queue.tail(10).copy()

print("Weakest 10 baseline picks:")
print(
    weak_picks[
        [
            "score",
            "reason_code",
            "action",
            "content_age_days",
            "impressions_90d",
            "avg_position"
        ]
    ].to_string(index=False)
)

future_or_label_columns = [
    "went_dark",
    "label",
    "target",
    "future_clicks",
    "future_impressions",
    "march_clicks",
    "march_impressions"
]

leaked_columns = [
    col for col in future_or_label_columns
    if col in df.columns
]

print("\nLeakage columns found:", leaked_columns)

assert len(leaked_columns) == 0

print("Leakage check: PASSED")

Weakest 10 baseline picks:
 score     reason_code         action  content_age_days  impressions_90d  avg_position
     0 GENERAL_REFRESH refresh_review                91                1           0.0
     0 GENERAL_REFRESH refresh_review                91                1           0.0
     0 GENERAL_REFRESH refresh_review                91                1           7.0
     0 GENERAL_REFRESH refresh_review                90                1           7.0
     0 GENERAL_REFRESH refresh_review                90                1           9.0
     0 GENERAL_REFRESH refresh_review                90                1           0.0
     0 GENERAL_REFRESH refresh_review                90                1           0.0
     0 GENERAL_REFRESH refresh_review                90                1           1.0
     0 GENERAL_REFRESH refresh_review                90                1           0.0
     0 GENERAL_REFRESH refresh_review                90                1           4.0

Leakage columns

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.